# 🚀 Imperative Migration Example: Past the Clean Baseline

This notebook demonstrates how to perform a **Manual Migration** using the high-integrity output of the `clean-dataset` CLI. 

In this approach, we ignore the `config.yaml` pipeline and use standard vectorized Pandas operations to perform domain-specific cleaning. This is the recommended path for users who want total control over their transformation sequence.

In [7]:
import pandas as pd
import numpy as np
from dd_cleaner.notebook_utils import init_notebook_session, get_cleaned_data

# 1. Initialize session and load the 'Clean Baseline'
# Point to the directory containing the 'data/' folder
coord, _ = init_notebook_session("..")
df = get_cleaned_data(coord)

print(f"Loaded Baseline: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

✅ Notebook session initialized for workspace: /home/rajiv/programming/dd-parser-cleaner/tests

Available Artifacts:

Artifact Name                                  File Name  \
0                         Raw Data                          sba_loans_raw.csv   
1                     Cleaned Data                    sba_loans_raw_clean.csv   
2                User Cleaned Data             sba_loans_raw_user_cleaned.csv   
3             Tagged Entities (DD)         sba_loans_raw_analysis_results.csv   
4  Cleaning Recommendations Report                cleaning_recommendations.md   
5                 Profiling Report          sba_loans_raw_profiling_report.md   
6                   Handshake File  sba_loans_raw_parser_cleaner_handshake.md   
7                  Quarantine File               sba_loans_raw_quarantine.csv   
8               Metadata Authority           sba_loans_raw_metadata_table.csv   

                                            Location  Exists  
0                             data/sba_loans_raw.csv    True  
1            data/dd_cleaner/sba_loans_raw_clean.csv    True  
2     data/dd_cleaner/sba_loans_raw_user_cleaned.csv    True  
3  documents/dd_analysis_results/sba_loans_raw_an...    True  
4   documents/dd_cleaner/cleaning_recommendations.md    True  
5  documents/dd_cleaner/sba_loans_raw_profiling_r...    True  
6  documents/dd_cleaner/sba_loans_raw_parser_clea...    True  
7       data/quarantine/sba_loans_raw_quarantine.csv   False  
8   data/dd_cleaner/sba_loans_raw_metadata_table.csv   False

Loaded Baseline: 1 rows, 43 columns


,AsOfDate,Program,LocationID,BorrName,BorrStreet,BorrCity,BorrState,BorrZip,BankName,BankFDICNumber,...,BusinessType,BusinessAge,LoanStatus,PaidInFullDate,ChargeOffDate,GrossChargeOffAmount,RevolverStatus,JobsSupported,CollateralInd,SoldSecMrktInd
0,2020-01-01,0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


## 1. Row-Transformation (Filtering)
Removing records based on domain-specific logic (e.g., date consistency).

In [8]:
# Example: Remove rows where First Disbursement is after Paid In Full
fdd = pd.to_datetime(df['FirstDisbursementDate'], errors='coerce')
pifd = pd.to_datetime(df['PaidInFullDate'], errors='coerce')

invalid_mask = fdd > pifd
df = df[~invalid_mask].copy()

print(f"Rows after date consistency filter: {len(df)} (Dropped {invalid_mask.sum()})")

Rows after date consistency filter: 1 (Dropped 0)


## 2. Attribute Management (Drop & Rename)
Removing technical noise and renaming columns for downstream readability.

In [9]:
# Drop unnecessary technical columns
cols_to_drop = ['AsOfDate', 'LocationID']
df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

# Rename attributes for clarity
rename_map = {
    'BorrName': 'borrower_name',
    'GrossApproval': 'total_loan_amount'
}
df.rename(columns=rename_map, inplace=True)

print(f"Columns remaining: {list(df.columns[:5])}...")

Columns remaining: ['Program', 'borrower_name', 'BorrStreet', 'BorrCity', 'BorrState']...


## 3. Missing-Values (Imputation)
Filling NaNs with logical defaults based on the attribute's nature.

In [10]:
# Impute categorical with a sentinel string
df['BorrState'] = df['BorrState'].fillna('UNKNOWN')

# Impute numeric with a constant (e.g., 0 for charge-off amount if missing)
df['GrossChargeOffAmount'] = df['GrossChargeOffAmount'].fillna(0)

print("Imputation complete.")

Imputation complete.


## 4. Attribute Derivation (Feature Engineering)
Creating new analytical features from existing attributes.

In [11]:
# Derive 'loan_performance_ratio'
df['loan_performance_ratio'] = df['GrossChargeOffAmount'] / df['total_loan_amount']

# Create a boolean flag for 'is_distressed'
df['is_distressed'] = df['loan_performance_ratio'] > 0

df[['total_loan_amount', 'GrossChargeOffAmount', 'is_distressed']].head()

,total_loan_amount,GrossChargeOffAmount,is_distressed
0,0,0,False


## 5. The Unified Migration Chain (Best Practice Sequence)

To prevent clobbering dependencies, we use the following functional sequence. Note that **Renaming and Dropping** happen last to ensure all preceding logic has access to the full raw feature set.

In [12]:
def apply_migration_pipeline(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    A consolidated pipeline that respects data dependencies.
    """
    work_df = raw_df.copy()
    
    # 1. Row Filtering (Integrity First)
    fdd = pd.to_datetime(work_df['FirstDisbursementDate'], errors='coerce')
    pifd = pd.to_datetime(work_df['PaidInFullDate'], errors='coerce')
    work_df = work_df[~(fdd > pifd)].copy()
    
    # 2. Imputation (Fix holes before calculation)
    work_df['BorrState'] = work_df['BorrState'].fillna('UNKNOWN')
    work_df['GrossChargeOffAmount'] = work_df['GrossChargeOffAmount'].fillna(0)
    
    # 3. Derivation (Logic depends on original column names)
    # Note: We use 'GrossApproval' here because it hasn't been renamed yet
    work_df['loan_performance_ratio'] = work_df['GrossChargeOffAmount'] / work_df['GrossApproval']
    work_df['is_distressed'] = work_df['loan_performance_ratio'] > 0
    
    # 4. Schema Management (The terminal 'finishing' act)
    work_df.rename(columns={
        'BorrName': 'borrower_name',
        'GrossApproval': 'total_loan_amount'
    }, inplace=True)
    
    work_df.drop(columns=['AsOfDate', 'LocationID'], errors='ignore', inplace=True)
    
    return work_df

# Execute the chain
df_raw_baseline = get_cleaned_data(coord)
df_final = apply_migration_pipeline(df_raw_baseline)

print(f"Final Dataset Shape: {df_final.shape}")
df_final[['borrower_name', 'total_loan_amount', 'loan_performance_ratio']].head()

Final Dataset Shape: (1, 43)


,borrower_name,total_loan_amount,loan_performance_ratio
0,0,0,NaN
